## Building Unit Cleaning Code, for San Diego County

Used for joining with SDGE utility

In [45]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [46]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [47]:
# load parcels 
parcels = gpd.read_parquet("/../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

# filter for San Diego
parcels = parcels[parcels['County'] == "San Diego"]

In [48]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint

Access: https://sat-io.earthengine.app/view/gba

In [49]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

In [50]:
# load in SDG&E boundary map
sdge_shape = gpd.read_file("/../../capstone/electrigrid/data/utilities/IOU_shapefiles.geojson")

sdge_shape = sdge_shape[sdge_shape['Acronym'] == "SDG&E"]

## Unit-regression for multi-family homes

### Select parcels for multi-family homes

In [51]:
# only keep the geometry and parcel number of parcels
parcels = parcels[['PARNO', 'geometry']]

## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

### Investigative analysis
Some of the parcels have more than one Zillow point within them so let's add code to find where multiple points are within one polygon to add the units together before dropping the duplicate units. 

In [52]:
# Count points in polygons
zillow_in_parcels = (
        # Spatial join associates points and polygons that intersects each other
        parcels.sjoin(
            zillow_multi,
            how="inner",  # Only keep points that matches a polygon
        )
        .groupby('PARNO')  # Group points by polygons
        .size()  # Get number of points
        .rename('num_zillow_points')  # Name your column as you want it to appear in polygons
    )

# investigate the parcels where more than one zillow point exists
multiple_zillows = zillow_in_parcels[zillow_in_parcels > 1]
print(f"There are ", len(multiple_zillows)," parcels with more than one Zillow point within a parcel.")

There are  6945  parcels with more than one Zillow point within a parcel.


### Select buildings that are multi-family homes and add Zillow data
Adding the units_sum to the parcel_res accounts for the multiple Zillow point that can occur in one parcel. These values will be utilized going forward as we will be computing the regression for the total_volume of the buildings in one parcel against the total number of units within that parcel. 

In [53]:

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    predicate = "intersects")

building_zillow = building_zillow.drop(columns = "index_right")

In [54]:
parcels_res.head()

,PARNO,geometry,unit
7629947,4241720700,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,...",3.0
7630042,4241721300,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,...",2.0
7630043,4241722000,"MULTIPOLYGON Z (((-117.23549 32.79699 0.00000,...",3.0
7630075,4236921300,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0
7630078,4236220200,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,...",3.0


### Calculate the volume and aggregate based on parcels
The problem I think that's happening here is that the units are added to every single building. When multiple buildings are in one parcel the units are double counted. We need to add in the parcel number and either aggregate to the parcel.

In [55]:
# add in the parcel number to add all of the building volumes for the regression
building_zillow_parcels = building_zillow.sjoin(
                                parcels_res,
                                how = "left",
                                predicate = "intersects")

In [56]:
## FOR BUILDINGS INTERSECTING MORE THAN ONE PARCEL CALCULATE WHICH PARCEL IT INTERSECTS MORE
# change the crs to a projected crs
building_zillow_parcels = building_zillow_parcels.to_crs("EPSG:6933")
parcels_res = parcels_res.to_crs("EPSG:6933")

# calculate intersection area for each building-parcel pair
building_zillow_parcels["intersection_area"] = (
    building_zillow_parcels.geometry.values
    .intersection(parcels_res.loc[building_zillow_parcels["index_right"]].geometry.values).area)

# keep only the parcel with the largest overlap per building
building_zillow_parcels = (
    building_zillow_parcels
    .sort_values("intersection_area", ascending=False)
    .loc[~building_zillow_parcels.index.duplicated(keep="first")]
    .drop(columns="intersection_area"))

# drop the unnecessary columns
building_zillow_parcels = building_zillow_parcels.drop(columns = ['index_right', 'unit_left'])
building_zillow_parcels = building_zillow_parcels.rename(columns = {'unit_right': 'total_unit'})


In [57]:
building_zillow_parcels.head()

,source,id,height,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610297,146.0
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610346,146.0
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610218,146.0
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610223,146.0
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610273,146.0


In [58]:
# reproject data frame to crs with meters as units
building_m = building_zillow_parcels.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

In [59]:
building_m.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610297,146.0,47337.971098,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610346,146.0,47337.971098,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610218,146.0,47337.971098,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610223,146.0,47337.971098,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610273,146.0,47337.971098,321904.781097


In [60]:
# keep only observations with unit data
building_w_units = building_m[~building_m['total_unit'].isna()]

assert building_w_units['total_unit'].isna().sum() == 0

There are also some multi-family homes where the units equal zero. Let's drop those so that there we can also predict them based on the regression.

In [61]:
# drop multi-family homes where the total_unit is equal to zero
building_w_units = building_w_units.drop(building_w_units[building_w_units['total_unit'] == 0].index)

Aggregate the volumes for the unit regression by parcel.

In [62]:
# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

In [63]:
# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'PARNO')

In [64]:
building_w_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3,total_volume_m3
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610297,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610346,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610218,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610223,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610273,146.0,47337.971098,321904.781097,321904.781097


The parcel data is from 2014 while the Zillow data is from 2025. There are some Zillow points not in parcels which is assumed to be due to the differences in time frames from the data. Parcels were likely added after 2014. 

In [65]:
# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

In [66]:
building_w_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3,total_volume_m3
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610297,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610346,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610218,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610223,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610273,146.0,47337.971098,321904.781097,321904.781097


In [67]:
# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'PARNO', keep = 'first')

### Run the linear regression, remove outliers, and re-run the regression

In [68]:
# run model
results = smf.ols('total_unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

In [69]:
building_w_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3,total_volume_m3,residual
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610297,146.0,47337.971098,321904.781097,321904.781097,6.249568
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610346,146.0,47337.971098,321904.781097,321904.781097,6.249568
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610218,146.0,47337.971098,321904.781097,321904.781097,6.249568
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610223,146.0,47337.971098,321904.781097,321904.781097,6.249568
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610273,146.0,47337.971098,321904.781097,321904.781097,6.249568


In [70]:
# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

In [71]:
print(f"There are ", building_units_clean.shape[0], "buildings included in the regression.")
print(f"There are ", building_outliers.shape[0], "outliers")

There are  17647 buildings included in the regression.
There are  1421 outliers


In [72]:
# rerun linear regression
results_clean = smf.ols('total_unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

/tmp/ipykernel_2642838/860630456.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_2642838/860630456.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [73]:
intercept

24.96199909979423

In [74]:
slope

0.00035476594782909873

### Use the values from the linear regression to compute missing units

In [75]:
# extract just the multi-family homes where unit info is missing
missing_units = building_m[(building_m['total_unit'].isna()) & (building_m['total_unit'] == 0)]


# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'PARNO')

In [76]:
# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

In [77]:
# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

In [78]:
missing_outlier_units.head()

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3,total_volume_m3,residual,missing_total_volume_m3
7332898,ms,UnitedStates_023013221_543824,11.034998,10.497087,USA,"{'xmax': -117.15213851225344, 'xmin': -117.153...","POLYGON ((-11303727.646 3962197.494, -11303730...",Multi,2015.0,NaN,None,None,None,79046925.0,None,NaN,8355841,06073009203,h572,SDGE,RI104,6774001700,332.0,13687.238745,151038.651356,151038.651356,241.421085,151038.651356
7332899,ms,UnitedStates_023013221_96527,11.140231,10.459179,USA,"{'xmax': -117.15059105802244, 'xmin': -117.152...","POLYGON ((-11303553.782 3962073.844, -11303546...",Multi,2016.0,NaN,None,None,None,68526526.0,None,NaN,8355842,06073009203,h572,SDGE,RI104,6774001800,279.0,12631.413844,140716.869758,140716.869758,191.391467,140716.869758
7296551,osm,52976011,15.433562,9.272923,USA,"{'xmax': -117.21865589999999, 'xmin': -117.219...","POLYGON ((-11309992.084 3971943.221, -11309992...",Multi,2002.0,NaN,None,None,None,68143404.0,None,NaN,7890857,06073008343,h622,SDGE,RI104,3450724800,251.0,1610.743036,24859.502969,24859.502969,196.732671,24859.502969
6500638,ms,UnitedStates_023013203_292678,14.408304,7.853348,USA,"{'xmax': -117.3010475228742, 'xmin': -117.3016...","POLYGON ((-11317941.878 4005668.277, -11317943...",Multi,2020.0,NaN,None,None,None,110491913.0,None,NaN,8415569,06073019803,h491,SDGE,RI104,1670402100,342.0,1526.096528,21988.463037,24788.418429,287.753127,24788.418429
7015510,osm,1158251759,9.851967,3.173946,USA,"{'xmax': -117.01181570000001, 'xmin': -117.012...","POLYGON ((-11290100.810 3939966.583, -11290101...",Multi,2010.0,NaN,None,None,None,89367776.0,None,NaN,8340342,06073010018,h323,SDGE,RI104,6453900100,328.0,1500.206751,14779.987190,14779.987190,276.633334,14779.987190


In [79]:
# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('total_unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['total_unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

In [80]:
# range of calculated units
missing_outlier_units_pred['total_unit'].agg(['min','max'])

min    25.0
max    90.0
Name: total_unit, dtype: float64

In [81]:
## multiple buildings in one parcel have duplicated units since they were computed on total volume
#  only keep the first observation per parcel
missing_outlier_units_pred = missing_outlier_units_pred.drop_duplicates(subset = 'PARNO', keep = 'first')

In [82]:
# combine multi-family homes data frames
multi_complete = pd.concat([building_units_clean, missing_outlier_units_pred]).to_crs(zillow.crs)

# fill the total_volume for those with the missing_total_volume_m3 and rename to total_volume_m3
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# fill the total_volume for those with the missing_total_volume_m3 
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# drop unnecessary columns 
multi_complete = multi_complete.drop(['residual', 'geometry', 'total_volume_m3'], axis = 1)

# rename to total_volume_m3
multi_complete = multi_complete.rename(columns = {'missing_total_volume_m3' : 'total_volume_m3'})

# drop the unit data from parcel res as the new unit data was computed
parcels_res = parcels_res.drop(columns = ['unit'])

In [83]:
# join the parcel data on PARNO to get the parcel geometry
multi_by_parcel = parcels_res.merge(multi_complete, on = 'PARNO')

In [84]:
parcels_res.head()

,PARNO,geometry
7629947,4241720700,MULTIPOLYGON Z (((-11311567.091 3964494.557 0....
7630042,4241721300,MULTIPOLYGON Z (((-11311510.086 3964464.211 0....
7630043,4241722000,MULTIPOLYGON Z (((-11311616.383 3964438.185 0....
7630075,4236921300,MULTIPOLYGON Z (((-11313159.182 3961251.281 0....
7630078,4236220200,MULTIPOLYGON Z (((-11313244.013 3962367.862 0....


In [85]:
multi_complete.head()

,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,total_unit,area_m2,volume_m3,total_volume_m3
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610297,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610346,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610218,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610223,146.0,47337.971098,321904.781097,321904.781097
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...",Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7725610273,146.0,47337.971098,321904.781097,321904.781097


### Remove duplicates and make centroids for multi-family homes
Multi-family homes are in polygons. In order to combine the multi-family and single family data we need to make them points. The unit linear regression calculations were completed for buildings so there are multiples of each polygon. We only need one row per parcel to make the centroids. 

In [86]:
# check how many parcel numbers are duplicates
duplicates = multi_by_parcel.pivot_table(index = ['PARNO'], aggfunc = 'size')
print(f"There are ",duplicates[duplicates > 1].sum(), " duplicated parcels.")

There are  0  duplicated parcels.


**Remove duplicates so that there aren't duplicate points. The units are aggregated by parcel because they were run on the total_volume.**

In [87]:
# remove duplicates from the parcel number so we don't get multiple points per parcel
multi_by_parcel = multi_by_parcel.drop_duplicates(subset = 'PARNO', keep = 'first')

In [88]:
multi_by_parcel.head()

,PARNO,geometry,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3
0,4236921300,MULTIPOLYGON Z (((-11313159.182 3961251.281 0....,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368
1,4245420100,MULTIPOLYGON Z (((-11311377.586 3963837.438 0....,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,1755.586108
2,4245420700,MULTIPOLYGON Z (((-11311356.480 3963757.406 0....,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912
3,5712820100,MULTIPOLYGON Z (((-11297194.944 3945377.610 0....,ms,UnitedStates_023013221_591587,3.844132,0.142668,USA,"{'xmax': -117.08611408293011, 'xmin': -117.086...",Multi,1952.0,3.0,None,None,O,479668.0,living,1224.0,8230580,06073012600,h564,SDGE,RI101,2.0,233.630791,898.107589,898.107589
4,5681511000,MULTIPOLYGON Z (((-11296755.746 3947749.540 0....,osm,956699452,7.273522,0.659870,USA,"{'xmax': -117.08155640000001, 'xmin': -117.081...",Multi,1968.0,6.0,None,None,None,1150682.0,living,3290.0,8226851,06073012302,h577,SDGE,RI104,6.0,139.970610,1018.079362,1018.079362


In [89]:
# set the geometry to the parcel geometry
multi_by_parcel = multi_by_parcel.set_geometry('geometry')

# change the crs 
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

In [90]:
## IN ORDER TO CONCAT THE MULTI-FAMILY HOMES WITH SINGLE FAMILY HOMES THEY MUST ALL BE POINTS
# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid


In [91]:
multi_by_parcel.head()

,PARNO,geometry,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,centroids
0,4236921300,MULTIPOLYGON Z (((-11313159.182 3961251.281 0....,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,POINT (-11313153.723 3961239.229)
1,4245420100,MULTIPOLYGON Z (((-11311377.586 3963837.438 0....,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,1755.586108,POINT (-11311395.134 3963841.243)
2,4245420700,MULTIPOLYGON Z (((-11311356.480 3963757.406 0....,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,POINT (-11311373.489 3963745.624)
3,5712820100,MULTIPOLYGON Z (((-11297194.944 3945377.610 0....,ms,UnitedStates_023013221_591587,3.844132,0.142668,USA,"{'xmax': -117.08611408293011, 'xmin': -117.086...",Multi,1952.0,3.0,None,None,O,479668.0,living,1224.0,8230580,06073012600,h564,SDGE,RI101,2.0,233.630791,898.107589,898.107589,POINT (-11297213.003 3945383.249)
4,5681511000,MULTIPOLYGON Z (((-11296755.746 3947749.540 0....,osm,956699452,7.273522,0.659870,USA,"{'xmax': -117.08155640000001, 'xmin': -117.081...",Multi,1968.0,6.0,None,None,None,1150682.0,living,3290.0,8226851,06073012302,h577,SDGE,RI104,6.0,139.970610,1018.079362,1018.079362,POINT (-11296776.248 3947750.907)


In [92]:

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# rename the centroid to geometry
multi_by_parcel = multi_by_parcel.rename(columns = {'centroids' : 'geometry'})

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('geometry')

In [119]:
multi_by_parcel_processed.head()

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,geometry
0,4236921300,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,NaN,POINT (-11313153.723 3961239.229)
1,4236220200,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,3.0,117.733224,320.096607,320.096607,NaN,POINT (-11313238.717 3962356.234)
2,4245420100,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,3445.225886,NaN,POINT (-11311395.134 3963841.243)
3,4245420700,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,NaN,POINT (-11311373.489 3963745.624)
4,5670903600,osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,40.0,1895.486015,12525.523059,12525.523059,NaN,POINT (-11298091.345 3947127.302)


### Adjust single-family Zillow data to the SDG&E area and adjust values

In [93]:
# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

# subset for single family zillow and condos
single_zillow = zillow[zillow['type'] == "Single"].to_crs(zillow.crs)

# keep only single family homes within San Diego County
single_zillow = single_zillow.clip(sdge_shape)

# change all single_zillow to unit = 1
single_zillow['total_unit'] = 1

# drop the unit column
single_zillow = single_zillow.drop(['unit'], axis = 1)

## CALCULATE AREA FOR SINGLE FAMILY HOME
# select parcels where single family homes exist 
single_parcels = parcels.sjoin(single_zillow, predicate="contains").index.unique()
single_parcels = parcels.loc[single_parcels]

# crop to residential buildings (by keeping only those within single residential parcels)
single_buildings = building.sjoin(single_parcels, predicate="intersects").index.unique()
buildings_res = building.loc[single_buildings]

# keep all single-family buildings, and add zillow points only where they match up
single_building_zillow = gpd.sjoin(
    buildings_res,
    single_zillow,
    predicate = "intersects")

single_building_zillow = single_building_zillow.drop(columns = "index_right")

# reproject data frame to crs with meters as units
single_building_zillow = single_building_zillow.to_crs("EPSG:6933")

# create column from polygon area
single_building_zillow['area_m2'] = single_building_zillow.area

# rename height column to be clear about units
single_building_zillow.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
single_building_zillow['volume_m3'] = single_building_zillow['area_m2'] * single_building_zillow['height_m']



# concat the single and multifamily dataframes
complete_zillow = pd.concat([multi_by_parcel_processed, single_building_zillow], axis = 0)

ValueError: Cannot determine common CRS for concatenation inputs, got ['WGS 84', 'WGS 84 / NSIDC EASE-Grid 2.0 Global']. Use `to_crs()` to transform geometries to the same CRS before merging.

In [121]:
complete_zillow.head()

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,geometry,unit
0,4236921300,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,NaN,POINT (-117.25142 32.76728),NaN
1,4236220200,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,3.0,117.733224,320.096607,320.096607,NaN,POINT (-117.25230 32.77765),NaN
2,4245420100,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,3445.225886,NaN,POINT (-117.23320 32.79144),NaN
3,4245420700,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,NaN,POINT (-117.23297 32.79055),NaN
4,5670903600,osm,1182648717,6.608080,1.040565,USA,"{'xmax': -117.09509669999997, 'xmin': -117.095...",Multi,1968.0,NaN,None,None,None,5433899.0,living,25844.0,8226026,06073012501,h564,SDGE,RI104,40.0,1895.486015,12525.523059,12525.523059,NaN,POINT (-117.09531 32.63634),NaN


In [154]:
# ensure no total_unit values are na
complete_zillow[complete_zillow['total_unit'].isna()]

# ensure no total_unit values are below 1
complete_zillow[complete_zillow['total_unit']<1]

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,missing_total_volume_m3,geometry


In [58]:
print(f"Number of Multi-family homes that have a unit of zero:", len(complete_zillow[complete_zillow['total_unit']<1]))

Number of Multi-family homes that have a unit of zero: 0


## Final results

## Renaming and saving

In [59]:
# save the complete Zillow points
complete_zillow_sd = complete_zillow.copy()

In [ ]:
# takes a really long time :,(
complete_zillow_sd.to_file("data/complete_zillow_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 